# Critical Ising Power-Law Observable Scan

This notebook plots the new direct-puMPS scan for the two code subspaces `I_sigma` and `I_epsilon`. It reads the three base-2 observable CSVs, prints the state-quality diagnostics, overlays the log-log power-law fits, and saves PDF/PNG figures.


In [3]:
from pathlib import Path
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")
import csv
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import NullFormatter, NullLocator, FuncFormatter

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman", "CMU Serif", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "font.size": 20,
    "axes.labelsize": 24,
    "axes.titlesize": 26,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 18,
    "lines.linewidth": 3.0,
    "lines.markersize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "axes.linewidth": 1.5,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
})

marker_cycle = ["o", "s", "^", "D", "v", "P", "X"]

root = Path.cwd()
if not (root / "examples").exists():
    root = root.parent
examples_dir = root / "examples"

raw_files = {
    "small_avg_relent_X": examples_dir / "ising_powerlaw_small_avg_relent_X.csv",
    "complement_proxy": examples_dir / "ising_powerlaw_complement_proxy.csv",
    "mutual_information_XR": examples_dir / "ising_powerlaw_mutual_information_XR.csv",
}
fit_path = examples_dir / "ising_powerlaw_fit_summary.csv"
FIT_X_MAX = 0.25
FIT_TOL = 1e-14
CODE_LABELS = ["I_sigma", "I_epsilon"]

rows = []
for quantity, path in raw_files.items():
    with path.open() as f:
        reader = csv.DictReader(f)
        for r in reader:
            rows.append({
                "L": int(r["L"]),
                "bond_dim": int(r["bond_dim"]),
                "ell": int(r["ell"]),
                "x": float(r["x"]),
                "code_label": r["code_label"],
                "quantity": r["quantity"],
                "value_bits": float(r["value_bits"]),
                "gap_ratio": float(r["gap_ratio"]),
                "P0": float(r["P0"]),
                "P1": float(r["P1"]),
                "P2": float(r["P2"]),
                "method": r["method"],
            })

def fit_power_law_notebook(all_rows, quantity, code_label):
    selected = [
        r for r in all_rows
        if r["quantity"] == quantity
        and r["code_label"] == code_label
        and r["x"] <= FIT_X_MAX
        and r["value_bits"] > FIT_TOL
    ]
    if len(selected) < 2:
        raise ValueError(f"Not enough fit points for {quantity}, {code_label}")
    x = np.array([r["x"] for r in selected], dtype=float)
    y = np.array([r["value_bits"] for r in selected], dtype=float)
    coeff = np.polyfit(np.log2(x), np.log2(y), 1)
    exponent = float(coeff[0])
    intercept = float(coeff[1])
    pred = intercept + exponent * np.log2(x)
    target = np.log2(y)
    ss_res = float(np.sum((target - pred) ** 2))
    ss_tot = float(np.sum((target - np.mean(target)) ** 2))
    r2 = 1.0 if ss_tot == 0 else 1.0 - ss_res / ss_tot
    return {
        "quantity": quantity,
        "code_label": code_label,
        "prefactor_c": float(2.0 ** intercept),
        "exponent": exponent,
        "intercept_log2": intercept,
        "n_points": len(selected),
        "x_min": float(np.min(x)),
        "x_max": float(np.max(x)),
        "r2": r2,
        "fit_rule": f"x <= {FIT_X_MAX} and value_bits > {FIT_TOL}",
    }

fits = []
for quantity in raw_files:
    for code_label in CODE_LABELS:
        fits.append(fit_power_law_notebook(rows, quantity, code_label))

fit_columns = [
    "quantity", "code_label", "prefactor_c", "exponent", "intercept_log2",
    "n_points", "x_min", "x_max", "r2", "fit_rule",
]
with fit_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fit_columns)
    writer.writeheader()
    for fit in sorted(fits, key=lambda r: (r["quantity"], r["code_label"])):
        writer.writerow({k: fit[k] for k in fit_columns})

print(f"loaded {len(rows)} raw rows from {examples_dir}")
print(f"computed {len(fits)} fit rows with x <= {FIT_X_MAX}; saved {fit_path}")
print("\nquality summary:")
seen = set()
for r in sorted(rows, key=lambda z: (z["L"], z["ell"], z["code_label"], z["quantity"])):
    if r["L"] in seen:
        continue
    seen.add(r["L"])
    print(f"L={r['L']:2d}, D={r['bond_dim']:2d}, gap_ratio={r['gap_ratio']:.6f}, parity=({r['P0']:.6f},{r['P1']:.6f},{r['P2']:.6f})")

quantity_specs = {
    "small_avg_relent_X": {
        "ylabel": r"$\frac{1}{D}\sum_a S(\rho_X^a\Vert\bar{\rho}_X)$",
        "filename": "ising_powerlaw_small_avg_relent_X",
        "coefficient_label": "c_1",
        "exponent_label": r"\alpha",
    },
    "complement_proxy": {
        "ylabel": r"$\log D-\frac{1}{D}\sum_a S(\rho_{\bar{X}}^a\Vert\bar{\rho}_{\bar{X}})$",
        "filename": "ising_powerlaw_complement_proxy",
        "coefficient_label": "c_2",
        "exponent_label": r"\beta",
    },
    "mutual_information_XR": {
        "ylabel": r"$I(X:R)_{|\Psi_{QR}\rangle}$",
        "filename": "ising_powerlaw_mutual_information_XR",
        "coefficient_label": "c_3",
        "exponent_label": r"\delta",
    },
}
code_specs = {
    "I_sigma": {"label": r"$I-\sigma$", "marker": marker_cycle[0], "color": "tab:blue"},
    "I_epsilon": {"label": r"$I-\epsilon$", "marker": marker_cycle[1], "color": "tab:red"},
}

for quantity, spec in quantity_specs.items():
    fig, ax = plt.subplots(figsize=(33.0, 13.5))
    qrows = [r for r in rows if r["quantity"] == quantity]
    annotation_lines = []

    for code_label, cinfo in code_specs.items():
        fit = next(f for f in fits if f["quantity"] == quantity and f["code_label"] == code_label)
        pts = sorted(
            [r for r in qrows if r["code_label"] == code_label and r["x"] <= fit["x_max"] and r["value_bits"] > 0],
            key=lambda r: (r["x"], r["L"], r["ell"]),
        )
        if not pts:
            continue
        x = np.array([p["x"] for p in pts], dtype=float)
        y = np.array([p["value_bits"] for p in pts], dtype=float)
        ax.plot(
            x,
            y,
            linestyle="None",
            marker=cinfo["marker"],
            color=cinfo["color"],
            label=cinfo["label"],
            markersize=25,
            alpha=0.95,
        )

        x_line = np.logspace(np.log10(fit["x_min"] * 0.95), np.log10(fit["x_max"] * 1.05), 300)
        y_line = fit["prefactor_c"] * (x_line ** fit["exponent"])
        ax.plot(x_line, y_line, linestyle="--", alpha=0.8, color=cinfo["color"])
        annotation_lines.append(
            rf"{cinfo['label']}: ${spec['exponent_label']}={fit['exponent']:.4f}$" + "\n" +
            rf"$ {spec['coefficient_label']}={fit['prefactor_c']:.4f}$"
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$x$", fontsize=75)
    ax.set_ylabel(spec["ylabel"], fontsize=60)
    ax.tick_params(axis="both", labelsize=60)
    ax.set_xticks([0.08, 0.1, 0.15, 0.2, 0.25])
    ax.set_xticklabels([r"$0.08$", r"$0.1$", r"$0.15$", r"$0.2$", r"$0.25$"], fontsize=60)
    ax.set_xlim(0.075, 0.265)
    ax.xaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: ("{:.3f}".format(val)).rstrip("0").rstrip(".")))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.grid(True, which="both")
    ax.legend(loc="upper left", fontsize=52, frameon=False)
    ax.text(
        0.50,
        0.06,
        "\n\n".join(annotation_lines),
        transform=ax.transAxes,
        fontsize=42,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8),
    )

    fig.suptitle("Chiral Ising Code", y=0.99, fontsize=68)
    fig.tight_layout()
    pdf_path = examples_dir / f"{spec['filename']}.pdf"
    png_path = examples_dir / f"{spec['filename']}.png"
    fig.savefig(pdf_path, dpi=1200, bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    print(f"saved {pdf_path}")
    print(f"saved {png_path}")
    plt.close(fig)

alpha_beta_quantities = ["small_avg_relent_X", "complement_proxy"]
fig, axes = plt.subplots(1, 2, figsize=(33, 13.5))

for ax, quantity in zip(axes, alpha_beta_quantities):
    spec = quantity_specs[quantity]
    qrows = [r for r in rows if r["quantity"] == quantity]
    annotation_lines = []

    for code_label, cinfo in code_specs.items():
        fit = next(f for f in fits if f["quantity"] == quantity and f["code_label"] == code_label)
        pts = sorted(
            [r for r in qrows if r["code_label"] == code_label and r["x"] <= fit["x_max"] and r["value_bits"] > FIT_TOL],
            key=lambda r: (r["x"], r["L"], r["ell"]),
        )
        if not pts:
            continue
        x = np.array([p["x"] for p in pts], dtype=float)
        y = np.array([p["value_bits"] for p in pts], dtype=float)
        ax.plot(
            x,
            y,
            linestyle="None",
            marker=cinfo["marker"],
            color=cinfo["color"],
            label=cinfo["label"],
            markersize=25,
            alpha=0.95,
        )

        x_line = np.logspace(np.log10(fit["x_min"] * 0.95), np.log10(fit["x_max"] * 1.05), 300)
        y_line = fit["prefactor_c"] * (x_line ** fit["exponent"])
        ax.plot(x_line, y_line, linestyle="--", alpha=0.8, color=cinfo["color"])
        annotation_lines.append(
            rf"{cinfo['label']}: ${spec['exponent_label']}={fit['exponent']:.4f}$" + "\n" +
            rf"$ {spec['coefficient_label']}={fit['prefactor_c']:.4f}$"
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$x$", fontsize=75)
    ax.set_ylabel(spec["ylabel"], fontsize=60)
    ax.tick_params(axis="both", labelsize=60)
    ax.set_xticks([0.08, 0.1, 0.15, 0.2, 0.25])
    ax.set_xticklabels([r"$0.08$", r"$0.1$", r"$0.15$", r"$0.2$", r"$0.25$"], fontsize=60)
    ax.set_xlim(0.075, 0.265)
    ax.xaxis.set_minor_locator(NullLocator())
    ax.xaxis.set_minor_formatter(NullFormatter())
    ax.yaxis.set_major_formatter(FuncFormatter(lambda val, _: ("{:.3f}".format(val)).rstrip("0").rstrip(".")))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.grid(True, which="both")
    ax.legend(loc="upper left", fontsize=52)
    ax.text(
        0.60,
        0.06,
        "\n\n".join(annotation_lines),
        transform=ax.transAxes,
        fontsize=45,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.8),
    )

fig.suptitle("Chiral Ising Code", y=0.99, fontsize=68)
fig.tight_layout()
combined_pdf_path = examples_dir / "ising_powerlaw_alpha_beta_quantities.pdf"
combined_png_path = examples_dir / "ising_powerlaw_alpha_beta_quantities.png"
fig.savefig(combined_pdf_path, dpi=1200, bbox_inches="tight")
fig.savefig(combined_png_path, dpi=300, bbox_inches="tight")
print(f"saved {combined_pdf_path}")
print(f"saved {combined_png_path}")
plt.close(fig)


loaded 210 raw rows from /Users/yuntaisong/codes/puMPS.jl/examples
computed 6 fit rows with x <= 0.25; saved /Users/yuntaisong/codes/puMPS.jl/examples/ising_powerlaw_fit_summary.csv

quality summary:
L=16, D=14, gap_ratio=7.980738, parity=(1.000000,-1.000000,1.000000)
L=18, D=16, gap_ratio=7.984794, parity=(1.000000,-1.000000,1.000000)
L=20, D=16, gap_ratio=7.987662, parity=(1.000000,-1.000000,1.000000)
L=22, D=18, gap_ratio=7.993651, parity=(1.000000,-0.999978,0.999822)
L=24, D=18, gap_ratio=7.991726, parity=(1.000000,-1.000000,0.999999)
saved /Users/yuntaisong/codes/puMPS.jl/examples/ising_powerlaw_small_avg_relent_X.pdf
saved /Users/yuntaisong/codes/puMPS.jl/examples/ising_powerlaw_small_avg_relent_X.png
saved /Users/yuntaisong/codes/puMPS.jl/examples/ising_powerlaw_complement_proxy.pdf
saved /Users/yuntaisong/codes/puMPS.jl/examples/ising_powerlaw_complement_proxy.png
saved /Users/yuntaisong/codes/puMPS.jl/examples/ising_powerlaw_mutual_information_XR.pdf
saved /Users/yuntaisong/co